# Adaptive Beamformer

In a previous section I went over how to make a **very basic** beamformer using the simple geometry of a phased array and the expected angle of arrival.

I also looked at the Capon/MVDR beamformer and showed how it improves over the basic beamformer and rejects interference spatially. But what do we do if the interference has **very** little spatial separation from the signal we want? Can we still use the phased array to recover the signal?

I was one of the primary developers for [**Hack-A-Sat 4**](https://hackasat.com/hackasat4/) and during our qualifying round I authored a challenge called ["Dark Side of the Dishy"](https://github.com/cromulencellc/hackasat-qualifier-2023/tree/main/challenges/2_Cant_Stop_The_Signal_Mal/3_Dark_Side_of_the_Dishy). In this challenge I presented the user with a signal from an environment where a 2 dimensional planar phased array had received two signals at the same time: one from a desired source, and one from a very high power interference source with almost zero spatial separation.

In this lab I'll focus on how we can use prior knowledge of part of the signal's contents to build a beamformer that recovers the whole signal, even in the presence of strong interference and without having to do an angular search.

Fun fact, one of our players said: "I thought I understood phased arrays until I played this CTF - I stand corrected"

That's always the sign of a great CTF challenge!

## Math

### Signal model

We use the same $8 \times 8$ planar array and steering vector $\mathbf{a}(az, el)$ from the MVDR lab. This time there are only two sources: the signal we want, $s[k]$, and a much stronger interferer, $i[k]$, sitting just a few degrees away. At sample $k$ the array gives us

$$
\mathbf{x}[k] = \mathbf{a}_s \, s[k] + \mathbf{a}_i \, i[k] + \mathbf{n}[k]
$$

where $\mathbf{a}_s$ and $\mathbf{a}_i$ are the steering vectors for the signal and the interferer. In `make_signal` the interferer is offset from the signal by `interferenceOffset` degrees in elevation, and is `isr` dB stronger than the signal.

Stacking $N$ samples as rows gives the $N \times Z$ data matrix that `make_signal` returns:

$$
\mathbf{X} = \begin{bmatrix} \mathbf{x}[0]^T \\ \mathbf{x}[1]^T \\ \vdots \\ \mathbf{x}[N-1]^T \end{bmatrix}
$$

### Why MVDR struggles here

MVDR minimizes output power while holding a gain of 1 in the look direction $\mathbf{a}_s$. When the interferer is very close, $\mathbf{a}_i \approx \mathbf{a}_s$, so the beamformer can't put a deep null on the interferer without also cutting into the look direction. The closer and louder the interferer, the worse this tradeoff gets. MVDR also needs to know $(az, el)$ of the signal ahead of time, and any error in that look direction makes things worse.

### Using what we already know: pilots

Most real signals start with something known: a preamble, a sync word, a header. In this lab the message always starts with `flag{`, so we know the first few characters, and therefore the first $P$ symbols that were sent. We call these known samples **pilots**, $\mathbf{p} = \begin{bmatrix} p[0] & \cdots & p[P-1] \end{bmatrix}^T$.

Instead of telling the beamformer *where* to look, we tell it *what* to look for. We want weights $\mathbf{w}$ so that the beamformer output during the pilots matches the pilots as closely as possible. With $\mathbf{Y}$ the first $P$ rows of $\mathbf{X}$:

$$
\min_{\mathbf{w}} \; \left\| \mathbf{Y}\mathbf{w} - \mathbf{p} \right\|^2
$$

This is an ordinary least squares problem, and its solution is

$$
\mathbf{w} = \left(\mathbf{Y}^H\mathbf{Y}\right)^{-1}\mathbf{Y}^H\mathbf{p}
$$

That's exactly what `adaptive_beamformer` computes, and `apply_beamformer` then applies those weights to **every** sample, $y[k] = \mathbf{x}[k]^T\mathbf{w}$, to recover the rest of the message.

### How this relates to MVDR

Look at the two pieces of the solution:
- $\frac{1}{P}\mathbf{Y}^H\mathbf{Y}$ is the spatial covariance $\mathbf{R}$, estimated from the pilot samples, just like in MVDR.
- $\frac{1}{P}\mathbf{Y}^H\mathbf{p}$ is the correlation between each antenna and the known signal. Since the pilots only correlate with the source that sent them, this is approximately $\mathbf{a}_s$ (up to a scale), estimated from the data.

So the adaptive weights are $\mathbf{w} \propto \mathbf{R}^{-1}\mathbf{a}_s$, the same shape as the MVDR weights. The difference is that MVDR assumes $\mathbf{a}_s$ from the geometry, while the adaptive beamformer **measures** it by correlating against the pilots. This is the classic Wiener filter / minimum mean square error (MMSE) beamformer. It doesn't need to know the array geometry, the calibration, or the direction of the source, only part of what it's sending.

### Things to keep in mind
- **Rank deficiency:** $\mathbf{Y}^H\mathbf{Y}$ is $Z \times Z$ but its rank is at most $P$. With one QPSK pilot character and 2 samples per symbol we only have $P = 8$ samples for $Z = 64$ antennas, so it can't be inverted. `adaptive_beamformer` uses the pseudo-inverse (`np.linalg.pinv`), which picks the smallest $\mathbf{w}$ that fits the pilots. Adding more pilot characters makes the estimate better.
- **Pilots must be aligned:** the pilots have to line up sample for sample with the start of the received signal. In a real receiver you would first find the preamble with a correlator, then train the weights.
- **Uncorrelated interference:** the interferer has to be sending something different from the pilots. If it were sending the same thing, the beamformer would happily keep both.
- **Static environment:** the weights are trained once at the start and then reused for the whole message. If the sources move, the weights need to be retrained (that's where algorithms like LMS and RLS come in).

## Things to notice

Play with the sliders below and compare the two constellations and decoded messages.

- Given enough spatial separation both beamformers work
- As the interference offset shrinks, the adaptive beamformer works **better**
- As the interference power increases, the adaptive beamformer works **better**
- The MVDR beamformer requires that we **know which angle to look at** and is **NOT** robust to pointing error
- The adaptive beamformer **requires zero knowledge about pointing** but rather knowledge about the signal itself



In [92]:
%pip install -q plotly nbformat ipywidgets


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [93]:
import numpy as np 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

In [94]:
messsage = "flag{HelloThisIsAMessageWeWantToRecover!!!}"

style = {"description_width": "160px"}
layout = widgets.Layout(width="500px")
encoding = widgets.Dropdown(options=["qpsk", "bpsk"], value="qpsk", description="Encoding",
                            style=style, layout=layout)
# number of pilot characters from the start of the message that we already know
nPilots = widgets.IntSlider(value=5, min=1, max=10, step=1, description="Pilot characters",
                            continuous_update=False, style=style, layout=layout)

signalAz = 60
signalEl = 45 

# elevation offset of the interferer from the signal of interest
interferenceOffset = widgets.FloatSlider(value=2.0, min=0.0, max=30.0, step=0.5, description="Interference offset (°)",
                                         continuous_update=False, style=style, layout=layout)
# elevation error in the direction MVDR is told to look, positive is toward the interferer
pointingError = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.25, description="MVDR pointing error (°)",
                                    continuous_update=False, style=style, layout=layout)
# samples per symbol
symbolRate = widgets.IntSlider(value=2, min=1, max=5, step=1, description="Samples per symbol",
                               continuous_update=False, style=style, layout=layout)

snrDb = 10.0 # Fixed per-antenna SNR of the signal vs. the receiver noise
# Interference power relative to the signal
isr = widgets.FloatSlider(value=0.0, min=-0.0, max=40.0, step=1.0, description="Interference/Signal (dB)",
                          continuous_update=False, style=style, layout=layout)

# Horizontal and vertical antenna positions in wavelengths. 8x8 like the MVDR lab, but with
# half wavelength spacing so the array is wide enough to separate sources a couple degrees apart
ANTENNA_V_GRID = np.arange(-2,2,0.5)
ANTENNA_H_GRID = np.arange(-2,2,0.5)

In [95]:
QPSK_SYMBOL_TABLE = [ 1+1j, -1+1j, -1-1j, 1-1j]
BPSK_SYMBOL_TABLE=  [1,-1]
SYMBOL_TABLES = { "qpsk": QPSK_SYMBOL_TABLE, "bpsk": BPSK_SYMBOL_TABLE }
BITS_PER_SYMBOL = { "qpsk": 2, "bpsk": 1 }

def encode_to_symbol( message: bytes , encoding: str, m: int  ):
    bdata = message.encode() if isinstance( message, str ) else message
    byte_array = bytearray( bdata )
    num_list = list( byte_array )
    symbol_table = SYMBOL_TABLES[ encoding ]
    b = BITS_PER_SYMBOL[ encoding ]
    mask = ( 1 << b ) - 1
    symbol_list=[]
    for num in num_list: 
        for i in range(8-b,-b,-b):
            bits =( num >> i ) & mask # Just take b bits bruh
            symbol = symbol_table[ bits ]
            symbol_list.append( symbol )
    symbolRepd = np.repeat( symbol_list , m )

    return symbolRepd

def makeQpskSignal( bytes, M , Abytes  ):
    symbolsRep = encode_to_symbol( bytes, "qpsk", M )
    S = ( Abytes * symbolsRep )
    return S
def makeBpskSignal( bytes, M, Abytes ): 
    symbolsRep = encode_to_symbol( bytes, "bpsk", M )
    S = ( Abytes * symbolsRep )
    return S

def decode_from_symbols( symbols , encoding: str, m: int ):
    symbols = np.asarray( symbols ).ravel()
    # Take the middle sample of each symbol
    centers = symbols[ m//2 :: m ]
    # Nearest point in the constellation gives the bits
    table = np.array( SYMBOL_TABLES[ encoding ] )
    bitsIdx = np.argmin( np.abs( centers[:,np.newaxis] - table[np.newaxis,:] ), axis=1 )
    b = BITS_PER_SYMBOL[ encoding ]
    symbolsPerByte = 8 // b
    byte_array = bytearray()
    for k in range( 0, len(bitsIdx) - symbolsPerByte + 1, symbolsPerByte ):
        num = 0
        for bits in bitsIdx[k:k+symbolsPerByte]:
            num = ( num << b ) | int(bits)
        byte_array.append( num )
    message = byte_array.decode( "ascii", errors="replace" )
    return message 

def apply_beamformer( signal , weights ):
    out = np.asarray( signal @ weights ).ravel()
    return out / np.max( np.abs( out ))

In [96]:
# Signal generator (from the MVDR lab)
locations = [] 
for V in ANTENNA_V_GRID:
    for H in ANTENNA_H_GRID: 
        loc = (H,V)
        locations.append(loc)
Z = len( locations )

def steering_vector( az, el ):
    # Path difference (in wavelengths) at each antenna. H is East and V is North
    azRad, elRad = np.radians( az ), np.radians( el )
    east  = np.cos(elRad) * np.sin(azRad)
    north = np.cos(elRad) * np.cos(azRad)
    locArray = np.array( locations )
    waveDelays = locArray[:,0] * east + locArray[:,1] * north
    return np.exp( 1j * 2*np.pi * waveDelays )

def make_signal( message, encoding, m, signalAz, signalEl, interferenceOffset, snrDb, isrDb, seed=0 ):
    rng = np.random.default_rng( seed )
    if encoding == "qpsk":
        s = makeQpskSignal( message, m, 1.0 )
    else:
        s = makeBpskSignal( message, m, 1.0 )
    N = len( s )

    # Noise power sets the SNR
    table = np.array( SYMBOL_TABLES[ encoding ] )
    signalPower = np.mean( np.abs( table )**2 )
    noisePower = signalPower * 10**( -snrDb / 10 )
    noiseStd = np.sqrt( noisePower )

    # High power interferer sending its own random symbols at the same symbol rate, I / S = ISR
    interference = 10**( isrDb / 20 ) * np.repeat( table[ rng.integers(0, len(table), N//m) ], m )

    # (N,Z) one set of N samples for each antenna
    signal = np.outer( s, steering_vector( signalAz, signalEl ) ) \
           + np.outer( interference, steering_vector( signalAz, signalEl + interferenceOffset ) )

    # Add complex white noise at each antenna. This also keeps the covariance invertible
    signal = signal + noiseStd/np.sqrt(2) * ( rng.standard_normal((N,Z)) + 1j*rng.standard_normal((N,Z)) )
    return signal


In [97]:
# Beamformers 

def adaptive_beamformer( pilots, signals ):
    nPilots = len(pilots)
    p = np.transpose( np.matrix( pilots )  ) 
    Y = np.matrix( signals[:nPilots,:] )   
    R = Y.H @ Y / nPilots
    # Ridge / diagonal loading: with about as many pilot samples as antennas plain least squares
    # fits the noise (and R is singular with fewer), so pull the solution toward small weights
    R = R + 0.1 * np.trace( R ).real / Z * np.eye( Z )
    w = np.linalg.solve( R, Y.H @ p / nPilots )
    return w 


def mvdr_2d_beamformer( az, el , signal , n ):
    # Spatial covariance from the first n samples, signal is (N,Z)
    X = np.matrix( signal[:n,:] ).T
    R = X @ X.H / n
    # Diagonal loading: the signal is in R and n is small, so without this MVDR cancels its own signal
    R = R + 0.1 * np.trace( R ).real / Z * np.eye( Z )
    Rinv = np.linalg.inv( R )
    a = np.matrix( steering_vector( az, el ) ).T
    w = ( Rinv @ a ) / ( a.H @ Rinv @ a )
    # Output is w^H x, apply_beamformer does signal @ weights so conjugate here
    return np.conj( w )

In [98]:
# Solvers
out = widgets.Output()

def update(_=None):
    m = symbolRate.value
    signal = make_signal( messsage, encoding.value, m, signalAz, signalEl, interferenceOffset.value,
                          snrDb, isr.value )

    pilots = encode_to_symbol( messsage[:nPilots.value] , encoding.value , m )
    wAdaptive = adaptive_beamformer( pilots, signal )
    wMvdr = mvdr_2d_beamformer( signalAz, signalEl + pointingError.value, signal, len(signal) )

    sMvdr = apply_beamformer( signal , wMvdr )
    sAdaptive = apply_beamformer( signal , wAdaptive )

    msgMvdr = decode_from_symbols( sMvdr  , encoding.value, m )
    msgAdaptive = decode_from_symbols( sAdaptive, encoding.value, m )

    fig = go.Figure()
    fig.add_trace( go.Scatter( x=sMvdr.real, y=sMvdr.imag, mode="markers", name="MVDR",
                               marker=dict(size=6, opacity=0.6) ) )
    fig.add_trace( go.Scatter( x=sAdaptive.real, y=sAdaptive.imag, mode="markers", name="Adaptive",
                               marker=dict(size=6, opacity=0.6) ) )
    fig.update_layout( title="Beamformer output (IQ)", width=550, height=550, template="plotly_white",
                       xaxis=dict(title="I", range=[-1.1,1.1]),
                       yaxis=dict(title="Q", range=[-1.1,1.1], scaleanchor="x", scaleratio=1) )

    # Gain of each beamformer over a window of az/el around the signal, normalized so its peak is 0 dB
    jammerEl = signalEl + interferenceOffset.value
    lookEl = signalEl + pointingError.value
    span = max( 20.0, interferenceOffset.value + 5, abs( pointingError.value ) + 5 )
    azGrid = np.linspace( signalAz - span, signalAz + span, 101 )
    elGrid = np.linspace( signalEl - span, signalEl + span, 101 )
    AZ, EL = np.meshgrid( np.radians( azGrid ), np.radians( elGrid ) )
    # same path difference as steering_vector, done for the whole grid at once -> (el, az, Z)
    locArray = np.array( locations )
    waveDelays = np.multiply.outer( np.cos(EL) * np.sin(AZ), locArray[:,0] ) \
               + np.multiply.outer( np.cos(EL) * np.cos(AZ), locArray[:,1] )
    gridSteering = np.exp( 1j * 2*np.pi * waveDelays )

    def pattern_db( w ):
        gain = np.abs( gridSteering @ np.asarray( w ).ravel() )
        return 20*np.log10( gain / gain.max() + 1e-12 )

    patternFig = make_subplots( rows=1, cols=2, horizontal_spacing=0.1,
                                subplot_titles=( "MVDR beam pattern", "Adaptive beam pattern" ) )
    for col, w in ( (1, wMvdr), (2, wAdaptive) ):
        patternFig.add_trace( go.Heatmap( x=azGrid, y=elGrid, z=pattern_db( w ), coloraxis="coloraxis" ), row=1, col=col )
        patternFig.add_trace( go.Scatter( x=[signalAz], y=[signalEl], mode="markers", name="Signal",
                                          legendgroup="signal", showlegend=( col == 1 ),
                                          marker=dict(symbol="star", size=16, color="white", line=dict(color="black", width=1)) ),
                              row=1, col=col )
        patternFig.add_trace( go.Scatter( x=[signalAz], y=[jammerEl], mode="markers", name="Jammer",
                                          legendgroup="jammer", showlegend=( col == 1 ),
                                          marker=dict(symbol="x", size=14, color="red", line=dict(color="black", width=1)) ),
                              row=1, col=col )
    # only MVDR is pointed, the adaptive beamformer never uses a direction
    patternFig.add_trace( go.Scatter( x=[signalAz], y=[lookEl], mode="markers", name="MVDR pointing",
                                      marker=dict(symbol="circle-open", size=18, color="orange", line=dict(width=3)) ),
                          row=1, col=1 )
    patternFig.update_layout( width=1100, height=550, template="plotly_white",
                              coloraxis=dict(colorscale="Viridis", cmin=-40, cmax=0, colorbar=dict(title="Gain (dB)")),
                              legend=dict(orientation="h", y=-0.15) )
    patternFig.update_xaxes( title="Azimuth (°)" )
    patternFig.update_yaxes( title="Elevation (°)" )
    # square pixels, and zooming one plot zooms the other
    patternFig.update_yaxes( scaleanchor="x", row=1, col=1 )
    patternFig.update_xaxes( matches="x", row=1, col=2 )
    patternFig.update_yaxes( matches="y", row=1, col=2 )
    with out:
        out.clear_output(wait=True)
        fig.show()
        patternFig.show()
        print( f"MVDR:     {msgMvdr}" )
        print( f"Adaptive: {msgAdaptive}" )

# drop callbacks from earlier runs of this cell so each slider move draws once
for s in (encoding, nPilots, interferenceOffset, pointingError, symbolRate, isr):
    s.unobserve_all(name="value")
    s.observe(update, names="value")

display(widgets.VBox([encoding, nPilots, interferenceOffset, pointingError, symbolRate, isr, out]))

update()